# MàthemaSharp.TimeSeries - consolidamento

## Usage

Esempi e verifiche in T sui sei formati. Prerequisiti: build Release e kernel JupyterNet eseguito sul runtime .NET 11. I confronti sono su casi piccoli: non certificano tutte le API o l'accuratezza generale. Nessuna dipendenza Charts o NDSharp.



In [ ]:
#load "../shared/MathemaSharp.Numerics.csx"
using System.Numerics;
using MathemaSharp.TimeSeries;
void Verify<T>() where T : IFloatingPointIeee754<T>, IMinMaxValue<T>
{
    T N(int n) => T.CreateChecked(n);
    var eps = T.BitIncrement(T.One)-T.One;
    var loess = Stl.Loess<T>([N(1),N(3),N(5),N(7),N(9)],[],5,N(2));
    if (T.Abs(loess-N(5)) > N(64)*eps*N(5)) throw new Exception("LOESS");
    var walk = Simulation.RandomWalk<T>(new Random(42),3,T.Zero,T.Zero,T.One,_ => T.One);
    if (walk[2] != N(3)) throw new Exception("Innovations");
    var fit = Garch11.FitVarianceTargeting<T>([T.One,-T.One,N(2),-N(2)],8,1);
    if (fit.Persistence >= T.One || !T.IsFinite(fit.Omega)) throw new Exception("GARCH");
    if (SpecialFunctions.RegularizedGammaQ(T.One,T.One) != T.Exp(-T.One)) throw new Exception("Gamma");
    Console.WriteLine($"{typeof(T).Name}: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK");
}
Verify<Half>();
Verify<float>();
Verify<double>();
Verify<Decimal32>();
Verify<Decimal64>();
Verify<Decimal128>();



Half: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK
Single: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK
Double: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK
Decimal32: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK
Decimal64: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK
Decimal128: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK

## Reference - API usate e famiglie correlate

| API | Constraint / limite |
|---|---|
| Stl.Loess | IEEE; ascisse centrate |
| Garch11.FitVarianceTargeting | IEEE; stimatore a griglia, non massimo globale garantito |
| Simulation + innovationSource | Innovazioni in T; distribuzione a cura del chiamante |
| Simulation default | Box-Muller double convertito in T |
| SpecialFunctions | IEEE; Lanczos a precisione limitata, non 34 cifre |

Questa tabella non sostituisce la Reference completa. [Usage](../../USAGE.md), [inventario API e limiti](../../ARCHITECTURE.md), [roadmap](../../ROADMAP.md). Gli output salvati testimoniano l'esecuzione di queste celle, non dei notebook precedenti.

